# **Bike Sharing Systems - Data Collection**


This notebook
1. scrapes a Wikipedia page listing global bicycle-sharing systems
to gather background data on each system, including location, launch date, and
fleet size.
2. collects current and forecasted weather data for cities of interest
using the OpenWeather API.

The output CSVs feed into the wrangling and analysis
steps of the pipeline.

Source: https://en.wikipedia.org/wiki/List_of_bicycle-sharing_systems

API docs: https://openweathermap.org/api

# **Webscraping**

In [46]:
# Import necessary libraries for webscraping

library(rvest)
library(httr)

## **Scrape the Wikipedia Table**

Read the page HTML, extract all table nodes, and identify the relevant one.

In [47]:
url <- "https://en.wikipedia.org/wiki/List_of_bicycle-sharing_systems"
root_node <- read_html(url)
table_nodes <- html_nodes(root_node, "table")
bike_df <- html_table(table_nodes[[1]], fill = TRUE)

## **Initial Data Summary**

A quick look at the structure and contents of the scraped data.

In [48]:
summary(bike_df)

   Country            Country          City / Region          Name          
 Length:897         Length:897         Length:897         Length:897        
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
    System            Operator           Launched         Discontinued      
 Length:897         Length:897         Length:897         Length:897        
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  

## **Export**

Save the raw data to CSV for use in the next step of the pipeline.

In [49]:
write.csv(bike_df, "raw_bike_sharing_systems.csv", row.names = FALSE)

# **API Calling**

## **Setup**

In [50]:
library(httr)

api_key <- "my_api_key"
base_url_current  <- "https://api.openweathermap.org/data/2.5/weather"
base_url_forecast <- "https://api.openweathermap.org/data/2.5/forecast"

## **Current Weather**

Fetch and display current weather for the cities of interest.

In [51]:
get_current_weather <- function(city_name) {
  response <- GET(base_url_current, query = list(
    q     = city_name,
    appid = api_key,
    units = "metric"
  ))

  r <- content(response, as = "parsed")

  data.frame(
    city       = city_name,
    weather    = r$weather[[1]]$main,
    visibility = ifelse(is.null(r$visibility), NA, r$visibility),
    temp       = r$main$temp,
    temp_min   = r$main$temp_min,
    temp_max   = r$main$temp_max,
    pressure   = r$main$pressure,
    humidity   = r$main$humidity,
    wind_speed = r$wind$speed,
    wind_deg   = r$wind$deg
  )
}

cities <- c("Seoul", "Washington, D.C.", "Paris", "Suzhou")
current_weather_df <- do.call(rbind, lapply(cities, get_current_weather))
summary(current_weather_df)

     city             weather            visibility         temp      
 Length:4           Length:4           Min.   :10000   Min.   : 8.76  
 Class :character   Class :character   1st Qu.:10000   1st Qu.:11.75  
 Mode  :character   Mode  :character   Median :10000   Median :13.63  
                                       Mean   :10000   Mean   :13.51  
                                       3rd Qu.:10000   3rd Qu.:15.39  
                                       Max.   :10000   Max.   :18.02  
    temp_min        temp_max        pressure       humidity       wind_speed   
 Min.   : 8.76   Min.   : 8.76   Min.   :1005   Min.   :47.00   Min.   :1.150  
 1st Qu.:11.75   1st Qu.:11.75   1st Qu.:1010   1st Qu.:47.75   1st Qu.:1.833  
 Median :13.09   Median :14.26   Median :1015   Median :68.50   Median :4.480  
 Mean   :12.20   Mean   :13.86   Mean   :1014   Mean   :69.25   Mean   :4.715  
 3rd Qu.:13.53   3rd Qu.:16.36   3rd Qu.:1018   3rd Qu.:90.00   3rd Qu.:7.362  
 Max.   :13.84   Max.  

## **5-Day Forecast**

Fetch 5-day, 3-hourly forecasts for a list of cities and combine into a
single data frame. Season is derived from the forecast month.

In [52]:
get_season <- function(month) {
  if (month %in% c(3, 4, 5))  return("Spring")
  if (month %in% c(6, 7, 8))  return("Summer")
  if (month %in% c(9, 10, 11)) return("Autumn")
  return("Winter")
}

get_weather_forecast_by_cities <- function(city_names) {
  all_rows <- list()

  for (city_name in city_names) {
    response <- GET(base_url_forecast, query = list(
      q     = city_name,
      appid = api_key,
      units = "metric"
    ))

    results <- content(response, as = "parsed")$list

    for (result in results) {
      month <- as.integer(format(as.POSIXct(result$dt_txt), "%m"))

      all_rows <- append(all_rows, list(data.frame(
        city              = city_name,
        weather           = result$weather[[1]]$main,
        visibility        = ifelse(is.null(result$visibility), NA, result$visibility),
        temp              = result$main$temp,
        temp_min          = result$main$temp_min,
        temp_max          = result$main$temp_max,
        pressure          = result$main$pressure,
        humidity          = result$main$humidity,
        wind_speed        = result$wind$speed,
        wind_deg          = result$wind$deg,
        forecast_datetime = result$dt_txt,
        season            = get_season(month)
      )))
    }
  }

  do.call(rbind, all_rows)
}

In [53]:
forecast_df <- get_weather_forecast_by_cities(cities)
summary(forecast_df)

     city             weather            visibility         temp      
 Length:160         Length:160         Min.   : 3690   Min.   : 4.11  
 Class :character   Class :character   1st Qu.:10000   1st Qu.:10.99  
 Mode  :character   Mode  :character   Median :10000   Median :14.09  
                                       Mean   : 9870   Mean   :14.15  
                                       3rd Qu.:10000   3rd Qu.:17.50  
                                       Max.   :10000   Max.   :24.29  
    temp_min        temp_max        pressure       humidity       wind_speed   
 Min.   : 4.11   Min.   : 4.11   Min.   : 999   Min.   :12.00   Min.   :0.110  
 1st Qu.:10.90   1st Qu.:11.14   1st Qu.:1012   1st Qu.:31.00   1st Qu.:1.738  
 Median :14.06   Median :14.09   Median :1016   Median :50.00   Median :2.445  
 Mean   :14.13   Mean   :14.18   Mean   :1016   Mean   :52.66   Mean   :2.765  
 3rd Qu.:17.50   3rd Qu.:17.50   3rd Qu.:1020   3rd Qu.:71.25   3rd Qu.:3.822  
 Max.   :24.29   Max.  

## **Supporting Datasets**

City reference data and the Seoul hourly bike demand dataset, used in
later analysis steps.

In [54]:
download.file(
  "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0321EN-SkillsNetwork/labs/datasets/raw_worldcities.csv",
  destfile = "raw_worldcities.csv"
)

download.file(
  "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0321EN-SkillsNetwork/labs/datasets/raw_seoul_bike_sharing.csv",
  destfile = "raw_seoul_bike_sharing.csv"
)

## **Export**

Save the raw data to CSV for use in the next step of the pipeline.

In [55]:
write.csv(forecast_df, "raw_cities_weather_forecast.csv", row.names = FALSE)